# Tiền xử lý dữ liệu WDI (2000–2025)

**Đồ án:** Đánh đổi giữa Tăng trưởng Kinh tế và Môi trường: Phân tích đường cong Kuznets và Sự chuyển dịch Năng lượng toàn cầu (2000 - 2022)

Notebook này triển khai **pipeline tiền xử lý dữ liệu** bám sát đặc tả trong `spec.md`, với mục tiêu duy nhất là chuẩn bị dữ liệu sạch và có cấu trúc tốt cho các bước phân tích/trực quan sau này.

Cụ thể, notebook sẽ:

1. Thiết lập môi trường, định nghĩa đường dẫn và cấu hình các tham số tiền xử lý.
2. Đọc `Data/Dataset.csv` → tạo bảng thô `df_raw` và kiểm tra nhanh.
3. Làm sạch dữ liệu → `df_clean` (xóa dòng rác, thay ".." bằng NaN, giới hạn năm 2000–2025, lọc đúng 10 series, chuyển kiểu dữ liệu).
4. Chuyển từ dạng rộng sang dạng dài (`df_long`) với mỗi dòng là 1 tổ hợp (Country, Series, Year).
5. Pivot sang dạng tidy (`df_tidy`) với 10 series là 10 cột, mỗi dòng là (Country, Year).
6. (Tùy chọn) Gợi ý bước bổ sung Region/Income nếu có file mapping.
7. Kiểm tra tính nhất quán và (tùy chọn) xuất file kết quả để dùng cho các notebook khác.

> **Lưu ý:** Notebook này được thiết kế để **vừa xây dựng vừa tự kiểm tra**, nên sau mỗi bước sẽ có cell hiển thị `shape`, vài dòng mẫu, và thống kê missing để đảm bảo pipeline hoạt động đúng như đặc tả.


---
## 1. Thiết lập môi trường và tham số tiền xử lý


In [ ]:
import pandas as pd
import numpy as np
import os
import re

# Đường dẫn thư mục Data (chạy từ thư mục gốc của project)
DATA_DIR = 'Data'
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'Data')

DATASET_PATH = os.path.join(DATA_DIR, 'Dataset.csv')
SERIES_METADATA_PATH = os.path.join(DATA_DIR, 'Series-Metadata.csv')  # chỉ dùng tham chiếu nếu cần

# Phạm vi năm theo spec
YEAR_MIN, YEAR_MAX = 2000, 2025

# Danh sách 10 Series Code đã chốt (theo spec.md)
SERIES_CODES = {
    'NY.GDP.PCAP.KD': 'gdp_per_capita',
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
    'EN.GHG.CO2.PC.CE.AR5': 'co2_per_capita',
    'EG.USE.PCAP.KG.OE': 'energy_use_per_capita',
    'EG.ELC.ACCS.ZS': 'access_electricity',
    'EG.CFT.ACCS.ZS': 'access_clean_fuels',
    'SP.POP.TOTL': 'population_total',
    'AG.LND.FRST.ZS': 'forest_area_percent',
    'EG.FEC.RNEW.ZS': 'renewable_energy_percent',
    'NV.IND.TOTL.ZS': 'industry_value_added_percent_gdp'
}

DESIRED_SERIES_LIST = list(SERIES_CODES.keys())

print("DATA_DIR:", DATA_DIR)
print("DATASET_PATH:", DATASET_PATH)
print("Số lượng Series Code mục tiêu:", len(DESIRED_SERIES_LIST))


---
## 2. Đọc Dataset.csv → df_raw và kiểm tra cấu trúc thô


In [ ]:
# Thử đọc Dataset.csv với một vài encoding phổ biến
encodings_to_try = ['utf-8', 'utf-8-sig', 'cp1252', 'latin-1']
df_raw = None
last_error = None

for enc in encodings_to_try:
    try:
        df_raw = pd.read_csv(DATASET_PATH, encoding=enc)
        print(f"Đọc Dataset.csv thành công với encoding: {enc}")
        break
    except Exception as e:
        last_error = e

if df_raw is None:
    raise RuntimeError(f"Không đọc được Dataset.csv với các encoding đã thử: {encodings_to_try}. Lỗi cuối: {last_error}")

print("Kích thước df_raw:", df_raw.shape)
print("Các cột đầu tiên:", list(df_raw.columns[:10]))

display(df_raw.head(5))
display(df_raw.tail(5))

# Kiểm tra nhanh xem cuối file có dòng metadata/văn bản hay không
print("Các giá trị duy nhất của cột 'Country Code' ở vài dòng cuối:")
display(df_raw.tail(10)[['Country Name', 'Country Code', 'Series Name', 'Series Code']])


---
## 3. Làm sạch dữ liệu → df_clean (xóa dòng rác, chuẩn hóa missing, lọc năm & series)


In [ ]:
df_clean = df_raw.copy()

# 3.1 Xóa các dòng hoàn toàn trống hoặc chỉ toàn NaN
before_drop_all_na = df_clean.shape[0]
df_clean = df_clean.dropna(how='all')
after_drop_all_na = df_clean.shape[0]
print(f"Số dòng bị xóa do toàn NaN: {before_drop_all_na - after_drop_all_na}")

# 3.2 Nhận diện và loại bỏ dòng metadata / không phải dữ liệu bảng
# Tiêu chí:
# - Country Code rỗng hoặc không có độ dài 3 ký tự chữ cái
# - HOẶC Series Code rỗng

def is_valid_country_code(code):
    if pd.isna(code):
        return False
    code = str(code).strip()
    return len(code) == 3 and code.isalpha()

def is_non_empty(value):
    if pd.isna(value):
        return False
    return str(value).strip() != ''

mask_valid_rows = df_clean.apply(
    lambda row: is_valid_country_code(row.get('Country Code')) and is_non_empty(row.get('Series Code')),
    axis=1
)

before_meta_filter = df_clean.shape[0]
df_clean = df_clean[mask_valid_rows].copy()
after_meta_filter = df_clean.shape[0]
print(f"Số dòng bị loại bỏ do Country Code/Series Code không hợp lệ (metadata/rác): {before_meta_filter - after_meta_filter}")

# 3.3 Xác định cột năm (2000–2025) dựa trên pattern 4 chữ số đầu cột
year_cols = []
for col in df_clean.columns:
    m = re.match(r'^(\d{4})', str(col))
    if m:
        year = int(m.group(1))
        if YEAR_MIN <= year <= YEAR_MAX:
            year_cols.append(col)

year_cols = sorted(year_cols, key=lambda c: int(str(c)[:4]))
print("Số cột năm chọn được (2000–2025):", len(year_cols))
print("Một vài cột năm đầu tiên:", year_cols[:5])

# 3.4 Thay thế chuỗi ".." bằng NaN chỉ trong các cột năm
df_clean[year_cols] = df_clean[year_cols].replace('..', np.nan)

# 3.5 Lọc theo 10 Series Code mục tiêu
existing_series = sorted(df_clean['Series Code'].dropna().unique())
print("Tổng số Series Code khác nhau trong dữ liệu sau khi lọc rác:", len(existing_series))

available_target_series = [s for s in DESIRED_SERIES_LIST if s in existing_series]
missing_target_series = [s for s in DESIRED_SERIES_LIST if s not in existing_series]

print("Series Code mục tiêu có trong dữ liệu:", available_target_series)
print("Series Code mục tiêu **không tìm thấy** trong dữ liệu:", missing_target_series)

df_clean = df_clean[df_clean['Series Code'].isin(available_target_series)].copy()
print("Kích thước df_clean sau khi chỉ giữ 10 series mục tiêu (những series có mặt):", df_clean.shape)

# 3.6 Chuyển các cột năm sang kiểu số (float)
for col in year_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print("Kiểu dữ liệu một vài cột năm sau khi convert:")
display(df_clean[year_cols[:5]].dtypes)

# 3.7 Thống kê missing theo Series Code để kiểm tra chất lượng dữ liệu
missing_by_series = (
    df_clean.groupby('Series Code')[year_cols]
    .apply(lambda x: x.isna().sum().sum())
    .sort_values(ascending=False)
)
print("Tổng số ô NaN theo từng Series Code (trên tất cả các năm):")
display(missing_by_series)

df_clean.head()


---
## 4. Chuyển sang dạng dài (Long) với `pd.melt` → df_long


In [ ]:
id_vars = ['Country Name', 'Country Code', 'Series Name', 'Series Code']

# Đảm bảo các cột id_vars tồn tại
for col in id_vars:
    if col not in df_clean.columns:
        raise KeyError(f"Thiếu cột bắt buộc trong df_clean: {col}")

df_long = pd.melt(
    df_clean,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='Year',
    value_name='Value'
)

# Chuyển cột Year từ dạng chuỗi '2015 [YR2015]' sang integer 2015
year_extracted = df_long['Year'].astype(str).str.extract(r'(\d{4})', expand=False)
df_long['Year'] = year_extracted.astype('Int64')  # Int64 (nullable integer) để vẫn giữ được NaN nếu có

print("Kích thước df_long:", df_long.shape)
print("Các cột trong df_long:", df_long.columns.tolist())
display(df_long.head(10))

# Kiểm tra một vài tổ hợp Country-Series-Year để chắc chắn melt đúng
sample_check = (
    df_long
    .dropna(subset=['Value'])
    .sample(10, random_state=42) if len(df_long.dropna(subset=['Value'])) > 10 else df_long.dropna(subset=['Value'])
)
print("Một vài dòng mẫu (không NaN) để kiểm tra giá trị sau melt:")
display(sample_check)


---
## 5. Pivot sang dạng tidy (mỗi Series là một cột) → df_tidy


In [ ]:
index_cols = ['Country Name', 'Country Code', 'Year']

for col in index_cols:
    if col not in df_long.columns:
        raise KeyError(f"Thiếu cột bắt buộc trong df_long: {col}")

df_tidy = df_long.pivot_table(
    index=index_cols,
    columns='Series Code',
    values='Value',
    aggfunc='first'
)

# Chuyển MultiIndex cột (Series Code) thành cột bình thường, rồi đổi tên theo mapping SERIES_CODES
df_tidy = df_tidy.reset_index()

rename_mapping = {code: short for code, short in SERIES_CODES.items() if code in df_tidy.columns}
df_tidy = df_tidy.rename(columns=rename_mapping)

print("Kích thước df_tidy:", df_tidy.shape)
print("Một vài cột đầu tiên trong df_tidy:")
print(df_tidy.columns.tolist()[:15])
display(df_tidy.head(10))

# Thống kê sơ bộ missing cho từng biến (cột) trong df_tidy
missing_by_column = df_tidy.isna().sum().sort_values(ascending=False)
print("Số lượng NaN theo từng cột trong df_tidy:")
display(missing_by_column)


---
## 6. (Tùy chọn) Bổ sung Region / Income nếu có file metadata quốc gia

Hiện tại pipeline mới sử dụng `Dataset.csv` làm nguồn chính. Nếu sau này có thêm file mapping (ví dụ `Country-Metadata.csv` chứa `Country Code`, `Region`, `Income_Group`), có thể merge như sau:

```python
country_meta = pd.read_csv('Data/Country-Metadata.csv')
df_tidy = df_tidy.merge(country_meta[['Country Code', 'Region', 'Income_Group']], on='Country Code', how='left')
df_long = df_long.merge(country_meta[['Country Code', 'Region', 'Income_Group']], on='Country Code', how='left')
```

Trong notebook này, chúng ta **chưa** thực hiện bước này vì chưa có file mapping tương ứng; phần này chỉ đóng vai trò gợi ý, bám sát đặc tả trong `spec.md`.


---
## 7. Kiểm tra cuối cùng và (tùy chọn) xuất dữ liệu đã xử lý


In [ ]:
print("Tóm tắt các bảng trong pipeline tiền xử lý:")
print("- df_raw  :", df_raw.shape, "(dữ liệu gốc sau khi đọc)")
print("- df_clean:", df_clean.shape, "(sau khi làm sạch, lọc năm 2000–2025 và 10 series mục tiêu)")
print("- df_long :", df_long.shape, "(sau melt, mỗi dòng = Country × Series × Year)")
print("- df_tidy :", df_tidy.shape, "(sau pivot, mỗi dòng = Country × Year, các series là cột)")

# Kiểm tra nhanh một vài quốc gia và năm quan trọng
countries_to_check = ['World', 'China', 'United States']
years_to_check = [2000, 2010, 2022, 2025]

print("\nMẫu df_tidy cho một vài quốc gia và năm quan trọng (nếu có trong dữ liệu):")
mask_sample = df_tidy['Country Name'].isin(countries_to_check) & df_tidy['Year'].isin(years_to_check)
display(df_tidy[mask_sample].sort_values(['Country Name', 'Year']))

# (Tùy chọn) Lưu ra file CSV để sử dụng ở các notebook khác
SAVE_PROCESSED = False  # đổi thành True nếu muốn lưu

if SAVE_PROCESSED:
    processed_long_path = os.path.join(DATA_DIR, 'processed_long.csv')
    processed_tidy_path = os.path.join(DATA_DIR, 'processed_tidy.csv')

    df_long.to_csv(processed_long_path, index=False)
    df_tidy.to_csv(processed_tidy_path, index=False)

    print("\nĐã lưu df_long →", processed_long_path)
    print("Đã lưu df_tidy →", processed_tidy_path)
